# Batch update: Vendor Contacts from CSV (Snowflake Notebook)

This notebook updates **only** the **Vendor Contacts** column on **EDLDB.SC_SANDBOX.VC_CPFR_VENDOR_EMAIL**, for rows whose **Vendor Number** appears in your upload file.

**Who this is for:** Operators who are comfortable running notebook cells step-by-step. You do not need to write SQL by hand; the notebook builds it for you.

**Before you run:**
1. Place **Unapplied_Vendors.csv** under the workspace folder **cpfr_uploads** (same pattern as other CPFR batch tools).
2. The CSV must contain columns named exactly **Vendor Number** and **Vendor Contacts** (extra columns are ignored). Column order does not matter.
3. Resolve any **duplicate Vendor Numbers** in the file before the update step; the notebook will stop if duplicates exist.
4. Set **CONFIRM_UPDATE = True** in the designated cell only after you have reviewed the *before* preview.

**Safety:** Rows not listed in the CSV are never updated. Columns other than **Vendor Contacts** are never modified.


In [ ]:
# =============================================================================
# CONFIGURATION
# =============================================================================
"""
Central settings for this run. Change only what your environment requires.

TARGET_FQN:
  Full name of the table we update (database.schema.table).

CSV_PATH:
  In Snowflake Workspaces, the notebook runs with a working directory that
  includes your project files. Uploads typically live under cpfr_uploads/.

KEY_COL / CONTACT_COL:
  These names must match the CSV headers and the Snowflake column names
  (including spaces). We select columns by name so the spreadsheet layout
  can vary as long as these two headers exist.
"""

from pathlib import Path
from datetime import datetime

RUN_TS = datetime.now().strftime("%Y%m%d_%H%M%S")

TARGET_DB = "EDLDB"
TARGET_SCHEMA = "SC_SANDBOX"
TARGET_TABLE = "VC_CPFR_VENDOR_EMAIL"
TARGET_FQN = f"{TARGET_DB}.{TARGET_SCHEMA}.{TARGET_TABLE}"

KEY_COL = "Vendor Number"
CONTACT_COL = "Vendor Contacts"

# Temp staging table in the current session (not persistent after session ends).
TEMP_STAGE_NAME = f"TMP_VC_CPFR_VENDOR_CONTACTS_UPDATE_{RUN_TS}"

CSV_PATH = Path.cwd() / "cpfr_uploads" / "Unapplied_Vendors.csv"

if not CSV_PATH.exists():
    raise FileNotFoundError(
        f"CSV not found: {CSV_PATH}\n"
        "Upload Unapplied_Vendors.csv to cpfr_uploads in this workspace."
    )

print("Target table     :", TARGET_FQN)
print("Temp stage name  :", TEMP_STAGE_NAME)
print("CSV path         :", CSV_PATH)
print("Key column       :", KEY_COL)
print("Contact column   :", CONTACT_COL)


In [ ]:
# =============================================================================
# SNOWFLAKE SESSION AND SQL HELPERS
# =============================================================================
"""
Snowflake Notebooks already authenticate you. We use Snowpark's
get_active_session() instead of username/password connectors.

Define sf_execute and sf_query_df BEFORE calling them (unlike some older
notebooks that call USE before defs).
"""

import pandas as pd
from snowflake.snowpark.context import get_active_session

session = get_active_session()


def sf_execute(sql: str) -> None:
    """Run DDL/DML or session SQL (USE, etc.)."""
    session.sql(sql).collect()


def sf_query_df(sql: str) -> pd.DataFrame:
    """Run a SELECT and return results as a pandas DataFrame."""
    return session.sql(sql).to_pandas()


sf_execute("USE ROLE SC_USER")
sf_execute("USE DATABASE EDLDB")
sf_execute("USE SCHEMA SC_SANDBOX")
# Uncomment if your role requires an explicit warehouse for writes:
# sf_execute("USE WAREHOUSE STREAMLIT_XS_WH")

sf_query_df(
    "SELECT CURRENT_USER() AS u, CURRENT_ROLE() AS r, "
    "CURRENT_DATABASE() AS d, CURRENT_SCHEMA() AS s, CURRENT_WAREHOUSE() AS w"
)


In [ ]:
# =============================================================================
# READ CSV (BY COLUMN NAME), NORMALIZE, SUBSET
# =============================================================================
"""
We read all fields as text to avoid pandas guessing numeric types for vendor IDs.

After stripping headers and cell text, we keep only KEY_COL and CONTACT_COL.
Any other columns in the file are ignored on purpose.
"""


def read_csv_as_strings(path: Path) -> pd.DataFrame:
    """Load CSV with every column as string dtype."""
    return pd.read_csv(path, dtype=str, keep_default_na=True)


def normalize_headers(df: pd.DataFrame) -> pd.DataFrame:
    out = df.copy()
    out.columns = [c.strip() for c in out.columns]
    return out


def strip_string_fields(df: pd.DataFrame) -> pd.DataFrame:
    out = df.copy()
    for col in out.columns:
        out[col] = out[col].apply(lambda x: x.strip() if isinstance(x, str) else x)
    return out


df_csv_raw = read_csv_as_strings(CSV_PATH)
df_csv = strip_string_fields(normalize_headers(df_csv_raw))

missing = [c for c in (KEY_COL, CONTACT_COL) if c not in df_csv.columns]
if missing:
    raise KeyError(
        f"CSV is missing required column(s): {missing}. "
        f"Found columns: {list(df_csv.columns)}"
    )

df_upload = df_csv[[KEY_COL, CONTACT_COL]].copy()
print("Rows loaded (before dropping bad keys):", len(df_upload))
df_csv_raw.head(10)


In [ ]:
# =============================================================================
# VALIDATION: BLANK KEYS, DUPLICATE VENDOR NUMBERS
# =============================================================================
"""
Blank Vendor Number rows cannot be joined safely; we remove them after reporting.

Duplicate Vendor Numbers would make the UPDATE ambiguous (which contact wins?).
We fail fast and show the duplicate rows so you can fix the spreadsheet.
"""


def find_key_duplicates(df: pd.DataFrame, key_col: str) -> pd.DataFrame:
    mask = df.duplicated(subset=[key_col], keep=False)
    return df.loc[mask].sort_values(key_col)


blank_key = df_upload[KEY_COL].isna() | (df_upload[KEY_COL].astype(str).str.strip() == "")
n_blank = int(blank_key.sum())
print("Rows with blank Vendor Number (will be dropped):", n_blank)
if n_blank:
    display(df_upload.loc[blank_key].head(20))

df_upload = df_upload.loc[~blank_key].copy()
df_upload[KEY_COL] = df_upload[KEY_COL].astype(str).str.strip()

df_dupes = find_key_duplicates(df_upload, KEY_COL)
print("Duplicate Vendor Number rows:", len(df_dupes))
if len(df_dupes) > 0:
    display(df_dupes)
    raise ValueError(
        "Duplicate Vendor Number values in CSV. Fix the file so each vendor "
        "appears once, then re-run from the read cell."
    )

print("Final row count for update:", len(df_upload))
df_upload.head(15)


In [ ]:
# =============================================================================
# STAGE: WRITE UPLOAD TO A SESSION TEMP TABLE
# =============================================================================
"""
Snowpark writes the small two-column frame to a temporary table. The UPDATE
statement in a later cell joins this stage to the real target table.

Only Vendor Number and Vendor Contacts exist in the stage; nothing else from
the CSV is loaded into Snowflake here.
"""

sp_stage = session.create_dataframe(df_upload)
sp_stage.write.mode("overwrite").save_as_table(TEMP_STAGE_NAME, table_type="temp")

sf_query_df(f"SELECT COUNT(*) AS stage_rows FROM {TEMP_STAGE_NAME}")
sf_query_df(f'SELECT * FROM {TEMP_STAGE_NAME} LIMIT 20')


In [ ]:
# =============================================================================
# BEFORE UPDATE: CURRENT TARGET VALUES FOR KEYS IN THE CSV
# =============================================================================
"""
INNER JOIN limits the preview to vendors that actually exist in the target
table. Vendors only in the CSV (not in DB) appear in the next query.
"""

sql_before = f'''
SELECT
  TRIM(t."{KEY_COL}") AS "{KEY_COL}",
  t."{CONTACT_COL}" AS contacts_before
FROM {TARGET_FQN} t
INNER JOIN {TEMP_STAGE_NAME} s
  ON TRIM(t."{KEY_COL}") = TRIM(s."{KEY_COL}")
ORDER BY 1
'''

df_before = sf_query_df(sql_before)
matched_cnt = len(df_before)
print("Target rows matched by CSV keys (will receive UPDATE):", matched_cnt)
display(df_before.head(30))

sql_not_in_target = f'''
SELECT DISTINCT TRIM(s."{KEY_COL}") AS "{KEY_COL}"
FROM {TEMP_STAGE_NAME} s
WHERE NOT EXISTS (
  SELECT 1 FROM {TARGET_FQN} t
  WHERE TRIM(t."{KEY_COL}") = TRIM(s."{KEY_COL}")
)
ORDER BY 1
'''

df_missing = sf_query_df(sql_not_in_target)
print("CSV vendor numbers with NO row in target (UPDATE skips these):", len(df_missing))
if len(df_missing) > 0:
    display(df_missing.head(50))


## Confirm before write

Review **contacts_before** and counts above. Then run the next cell and set **`CONFIRM_UPDATE = True`** before running the UPDATE cell.


In [ ]:
# Set to True only after reviewing the before-preview above.
CONFIRM_UPDATE = False


In [ ]:
# =============================================================================
# APPLY UPDATE (ONLY WHEN CONFIRM_UPDATE IS True)
# =============================================================================
"""
UPDATE ... FROM joins the real table to the temp stage on Vendor Number and
sets Vendor Contacts from the CSV. Unmatched target rows are untouched.
"""

if not CONFIRM_UPDATE:
    raise RuntimeError(
        "Set CONFIRM_UPDATE = True in the previous cell after you review "
        "the before-preview, then re-run this cell."
    )

update_sql = f"""
UPDATE {TARGET_FQN} t
SET t."{CONTACT_COL}" = s."{CONTACT_COL}"
FROM {TEMP_STAGE_NAME} s
WHERE TRIM(t."{KEY_COL}") = TRIM(s."{KEY_COL}")
"""

sf_execute(update_sql)

rows_updated = sf_query_df(
    "SELECT GET_DML_UPDATED_ROWS(LAST_QUERY_ID()) AS updated_rows"
)["updated_rows"].iloc[0]
print("Rows updated (Snowflake-reported):", rows_updated)


In [ ]:
# =============================================================================
# AFTER UPDATE: SAME JOIN FOR VISUAL COMPARISON
# =============================================================================
"""
Re-query the same keys. Compare to contacts_before; we also merge in pandas
for a side-by-side view when both frames exist.
"""

sql_after = f'''
SELECT
  TRIM(t."{KEY_COL}") AS "{KEY_COL}",
  t."{CONTACT_COL}" AS contacts_after
FROM {TARGET_FQN} t
INNER JOIN {TEMP_STAGE_NAME} s
  ON TRIM(t."{KEY_COL}") = TRIM(s."{KEY_COL}")
ORDER BY 1
'''

df_after = sf_query_df(sql_after)
display(df_after.head(30))

if matched_cnt > 0:
    compare = df_before.merge(df_after, on=KEY_COL, how="outer")
    compare.columns = [KEY_COL, "contacts_before", "contacts_after"]
    print("Side-by-side (first 25 rows):")
    display(compare.head(25))
else:
    print("No matched rows earlier; nothing to compare.")
